# 02d · NAS 2. tur: ReLU k3 ile kontrolün tam COCO'da eşit uzun ince ayarı (`nas_r2`)

**Soru (Furkan, 2026-10-12):** ReLU 1. turda (10k alt küme, 10 epoch) 416'da −5,5 puan kaybetti, ama aynı kernelde ORT CPU'da ≈ %20 hızlı (N4: ReLU k5 / SiLU k5; k3 kazancı 02c + Jetson ile önce ölçülür). Kayıp kalıcı mı, yoksa önceden eğitilmiş **SiLU** ağırlıklarının yeni aktivasyona uyum süresi mi? Yalnız ReLU'yu uzun eğitmek "fazladan eğitim aldı" itirazına açık olduğundan **iki kol birebir aynı tarifle** eğitilir:

| Kol | Koşu | Mimari |
|---|---|---|
| Kontrol | `silu_dfull_k3` | dokunulmamış YOLOX-Nano |
| Deney | `relu_dfull_k3` | tüm SiLU → ReLU (kernel 3, tam derinlik) |

Diğer adaylar (HardSwish, dreduced) uzun eğitime alınmaz: gecikme eğitimden bağımsızdır ve N4'te kontrole göre en fazla %3–4 hız kazançları var; mAP'lerini tamamen geri kazansalar bile anlamlı bir Pareto kazancı üretemezler. Uzun eğitim yalnız gecikme tavanı yüksek olan adaya verilir (çok aşamalı / multi-fidelity arama: eşit kısa eğitimle eleme → eşit uzun eğitimle doğrulama).

**Karar ölçütü (eğitimden önce sabitlendi, Furkan 2026-10-12):** son epoch'ta 416'da **Δ = mAP(ReLU) − mAP(kontrol) ≥ −1,0** (Karar 2/5 NAS aşaması bütçesi) → ReLU geçer ve seçim adayı olur; aksi halde bu arama uzayında orijinal Nano Pareto-optimal kabul edilir. Ara epoch'lar (2, 4, …, 10) karar için değil, farkın kapanıp kapanmadığını gösteren öğrenme eğrisi içindir.

**Tarif:** tam COCO train2017 (118 287 görsel), **12 epoch**, son 2 epoch mosaic'siz (`no_aug_epochs=1`, 1. turla aynı kural), batch 64, fp16, çok ölçekli (320–640), ısınma 1 epoch, **lr 0,001 / 64 görsel** (1. turda kontrolü bozmadığı gösterildi), EMA, önceden eğitilmiş `yolox_nano.pth`'den. 1. tura göre yalnız veri (10k → tam train2017) ve epoch (10 → 12) değişir.

**Değerlendirme:** 1. turla aynı protokol — ayrı **fp32** `eval.py`, val2017 (5000), batch 64, conf 0,001, NMS 0,65, EMA ağırlığı. Ara epoch'lar (2, 4, 6, 8, 10) yalnız 416'da; son epoch (12) 320/416/512'de.

**Tahmini süre/maliyet** (P0 ölçümünden türetildi, ölçülmedi): kararlı epoch ≈ 69 sn × 11,8 ≈ 13,6 dk → kol başına ≈ 2,7 sa; değerlendirme kol başına 8 × ≈ 1,5 dk; toplam ≈ 6 sa ≈ 9–10 CU (L4, 1,54 CU/sa). Veri hazırlığı her yeni runtime'da ≈ 10–20 dk (train2017.zip ≈ 18 GB, resmi COCO sunucusundan Colab yereline).

**Kesintide:** runtime'ı yeniden bağla → hücre 1–6 (hücre 4'te Drive elle; hücre 5 veriyi yeniden indirir) → hücre 9 ve 10 (7 ve 8 gerekmez). Bitmiş koşular atlanır; yarım koşu Drive'daki son epoch checkpoint'inden `--resume` ile devam eder; ara anlık görüntüler Drive'da kalır.

**Kayıtlar** (`Drive edge_ai/results/nas_r2/`, depoda `results/nas_r2/`, `.pth` hariç): `manifest.json` · `runs/<koşu>/` (`config.json`, `status.json`, `train_log.txt`, `eval_e<epoch>_s<çöz>_log.txt`; `epoch<N>_ckpt.pth`, `latest_ckpt.pth`, `last_epoch_ckpt.pth` yalnız Drive'da) · `curve.csv` (koşu × epoch × çözünürlük) · `nas_r2_summary.json` (karar) · `nas_r2_curve.png`.

Kod: `exps/nas/nano_macro_long.py` (tarif + anlık görüntü kancası; mimari cerrahisi `nano_macro.py`'den), `scripts/nas_long.py` (koşular, değerlendirme planı, eğri, karar), `scripts/nas_search.py`. YOLOX fork'una dokunulmaz.

In [ ]:
# 1 · Ortam ve sürüm kaydı
import json, platform, subprocess, sys, datetime, os

def sh(cmd):
    try:
        return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()
    except Exception as e:
        return f"ERR {e}"

env = {
    "timestamp_utc": datetime.datetime.now(datetime.UTC).isoformat(timespec="seconds"),
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "gpu": sh("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader") or "none",
    "cuda_nvcc": sh("nvcc --version | tail -1"),
    "cpu_count": os.cpu_count(),
    "ram_gb": round(os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9, 1),
    "disk_free_content": sh("df -h /content | tail -1"),
}
try:
    import torch, torchvision
    env.update(torch=torch.__version__, torchvision=torchvision.__version__,
               torch_cuda=torch.version.cuda, cuda_available=torch.cuda.is_available())
except ImportError:
    env["torch"] = "not installed"
print(json.dumps(env, indent=2))

In [ ]:
# 2 · Kod: depo + YOLOX fork sabit commit (exp ve betik depoda: exps/nas, scripts)
REPO_URL = "https://github.com/Bey3fendi/edge-ai-object-detection.git"
YOLOX_COMMIT = "3ec6263"
WS = "/content/edge_ai_workspace"

import os, subprocess
if not os.path.isdir(WS):
    subprocess.run(["git", "clone", "--recurse-submodules", REPO_URL, WS], check=True)
else:  # runtime aynı kaldıysa depodaki son exp/betik sürümünü al
    subprocess.run(["git", "-C", WS, "pull", "-q", "--ff-only"], check=True)
subprocess.run(["git", "-C", f"{WS}/YOLOX", "fetch", "--all", "-q"], check=True)
subprocess.run(["git", "-C", f"{WS}/YOLOX", "checkout", "-q", YOLOX_COMMIT], check=True)

env["repo_commit"] = sh(f"git -C {WS} rev-parse --short HEAD")
env["yolox_commit"] = sh(f"git -C {WS}/YOLOX rev-parse --short HEAD")
env["yolox_dirty"] = bool(sh(f"git -C {WS}/YOLOX status --porcelain"))
print({k: env[k] for k in ("repo_commit", "yolox_commit", "yolox_dirty")})
assert env["yolox_commit"].startswith(YOLOX_COMMIT)
env["repo_dirty"] = bool(sh(f"git -C {WS} status --porcelain"))
assert not env["repo_dirty"], "depo kirli: kayıtlı commit koşulan kodu temsil etmez"

In [ ]:
# 3 · YOLOX bağımlılıkları
# Not: `pip install -e .` (setup.py develop) güncel setuptools + py3.13'te kırılıyor; YOLOX saf Python
# olduğu için kurulum yerine PYTHONPATH kullanılır. onnx-simplifier eğitimde gerekmez → atlandı.
!pip install -q loguru tabulate thop ninja pycocotools psutil tensorboard opencv-python-headless

import sys, os
YOLOX_DIR = f"{WS}/YOLOX"
if YOLOX_DIR not in sys.path:
    sys.path.insert(0, YOLOX_DIR)
os.environ["PYTHONPATH"] = YOLOX_DIR + ":" + os.environ.get("PYTHONPATH", "")

import importlib.metadata as md
for p in ["torch", "torchvision", "numpy", "opencv-python-headless", "pycocotools", "loguru", "thop"]:
    try:
        env[f"pkg_{p}"] = md.version(p)
    except md.PackageNotFoundError:
        env[f"pkg_{p}"] = None
import yolox
from yolox.exp import get_exp
exp = get_exp(None, "yolox-nano")
print("yolox import OK:", yolox.__file__)
print({k: v for k, v in env.items() if k.startswith("pkg_")})
SCRIPTS = f"{WS}/scripts"
if SCRIPTS not in sys.path:
    sys.path.insert(0, SCRIPTS)
import importlib, nas_search as ns, nas_long as nl
importlib.reload(ns); importlib.reload(nl)
EXP_FILE = f"{WS}/exps/nas/nano_macro_long.py"
BASE_EXP_FILE = f"{WS}/exps/nas/nano_macro.py"   # mimari cerrahisi buradan (nano_macro_long içe aktarır)

In [ ]:
# 4 · Drive + ağırlık (SHA256)
# İlk mount tarayıcıda kullanıcı onayı ister; bağlıysa tekrar denemez.
env.pop("pkg_yolox", None)
if not os.path.ismount("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/edge_ai"
DRIVE_DATA = f"{DRIVE_ROOT}/data"
DRIVE_WEIGHTS = f"{DRIVE_ROOT}/weights"
ROUND = nl.ROUND    # nas_r2: 2. tur, ReLU k3 + kontrol, tam COCO 12 epoch (Furkan, 2026-10-12)
DRIVE_RESULTS = f"{DRIVE_ROOT}/results/{ROUND}"
for d in (DRIVE_DATA, DRIVE_WEIGHTS, DRIVE_RESULTS):
    os.makedirs(d, exist_ok=True)

import hashlib, urllib.request, shutil
WEIGHT_URL = "https://github.com/Megvii-BaseDetection/YOLOX/releases/download/0.1.1rc0/yolox_nano.pth"
w_drive = f"{DRIVE_WEIGHTS}/yolox_nano.pth"
if not os.path.exists(w_drive):
    urllib.request.urlretrieve(WEIGHT_URL, w_drive)
WEIGHTS = "/content/yolox_nano.pth"
shutil.copy(w_drive, WEIGHTS)

h = hashlib.sha256()
with open(WEIGHTS, "rb") as f:
    for chunk in iter(lambda: f.read(1 << 20), b""):
        h.update(chunk)
env["weights_url"] = WEIGHT_URL
env["weights_sha256"] = h.hexdigest()
env["weights_bytes"] = os.path.getsize(WEIGHTS)
print(env["weights_sha256"], env["weights_bytes"])

In [ ]:
# 5 · Veri: tam COCO train2017 + val2017 (Colab yereline)
# train2017 görüntüleri Drive'da yok: Drive'da `data/train2017.zip` varsa oradan, yoksa resmi COCO sunucusundan
# Colab yereline indirilir (≈18 GB). Drive kotası için zip varsayılan olarak Drive'a KAYDEDİLMEZ; her yeni
# runtime'da yeniden indirilir. Anotasyonlar ve val2017 P0'dan beri Drive'da.
# Duman testi (hücre 8) için 10k alt kümenin yalnız anotasyon json'u açılır (görselleri train2017'de zaten var).
import zipfile, shutil, time
COCO_LOCAL = "/content/datasets/COCO"
ANN_ZIP = f"{DRIVE_DATA}/annotations_trainval2017.zip"
VAL_ZIP = f"{DRIVE_DATA}/val2017.zip"
SUBSET_ZIP = f"{DRIVE_DATA}/coco_train2017_10k_stratified.zip"
TRAIN_ZIP_DRIVE = f"{DRIVE_DATA}/train2017.zip"
TRAIN_ZIP_URL = "http://images.cocodataset.org/zips/train2017.zip"
SAVE_TRAIN_ZIP_TO_DRIVE = False   # True: ilk indirmede Drive'a da kopyalar (≈18 GB kota)
N_TRAIN = 118_287
for z in (ANN_ZIP, VAL_ZIP, SUBSET_ZIP):
    assert os.path.exists(z), f"{z} yok: önce 00 pilot notebook'unun hücre 5 ve 9'u"
t0 = time.time()
os.makedirs(f"{COCO_LOCAL}/annotations", exist_ok=True)
with zipfile.ZipFile(ANN_ZIP) as z:
    for name in ("annotations/instances_train2017.json", "annotations/instances_val2017.json"):
        if not os.path.exists(f"{COCO_LOCAL}/{name}"):
            z.extract(name, COCO_LOCAL)
with zipfile.ZipFile(SUBSET_ZIP) as z:
    if not os.path.exists(f"{COCO_LOCAL}/annotations/instances_train2017_10k.json"):
        z.extract("annotations/instances_train2017_10k.json", COCO_LOCAL)
if not os.path.isdir(f"{COCO_LOCAL}/val2017"):
    shutil.copy(VAL_ZIP, "/content/val2017.zip")
    with zipfile.ZipFile("/content/val2017.zip") as z:
        z.extractall(COCO_LOCAL)

n_local = len(os.listdir(f"{COCO_LOCAL}/train2017")) if os.path.isdir(f"{COCO_LOCAL}/train2017") else 0
if n_local != N_TRAIN:
    local_zip = "/content/train2017.zip"
    if os.path.exists(TRAIN_ZIP_DRIVE):
        print("train2017.zip Drive'dan kopyalanıyor…", flush=True)
        shutil.copy(TRAIN_ZIP_DRIVE, local_zip)
        env["train_zip_source"] = "drive"
    else:
        print("train2017.zip resmi COCO sunucusundan indiriliyor (≈18 GB)…", flush=True)
        subprocess.run(["wget", "-q", "--show-progress", "--progress=dot:giga", "-O", local_zip + ".part",
                        TRAIN_ZIP_URL], check=True)
        os.replace(local_zip + ".part", local_zip)
        env["train_zip_source"] = TRAIN_ZIP_URL
    env["train_zip_bytes"] = os.path.getsize(local_zip)
    env["train_zip_sha256"] = ns.sha256(local_zip)
    if SAVE_TRAIN_ZIP_TO_DRIVE and not os.path.exists(TRAIN_ZIP_DRIVE):
        shutil.copy(local_zip, TRAIN_ZIP_DRIVE + ".tmp")
        os.replace(TRAIN_ZIP_DRIVE + ".tmp", TRAIN_ZIP_DRIVE)
    print("açılıyor…", flush=True)
    subprocess.run(["unzip", "-q", "-o", local_zip, "-d", COCO_LOCAL], check=True)
    os.remove(local_zip)
env["train_images"] = len(os.listdir(f"{COCO_LOCAL}/train2017"))
env["val_images"] = len(os.listdir(f"{COCO_LOCAL}/val2017"))
env["train_ann_sha256"] = ns.sha256(f"{COCO_LOCAL}/annotations/instances_train2017.json")
env["data_prep_s"] = round(time.time() - t0, 1)
env["disk_free_after_data"] = sh("df -h /content | tail -1")
print({k: env.get(k) for k in ("train_images", "val_images", "train_zip_source", "train_zip_sha256",
                               "train_ann_sha256", "data_prep_s", "disk_free_after_data")})
assert env["train_images"] == N_TRAIN and env["val_images"] == 5000

In [ ]:
# 6 · Manifest: turun kimlik kartı (tarif, koşular, değerlendirme planı, karar kuralı, sürümler, hash'ler)
# Varsa parmak izi alanları karşılaştırılır; tur ortasında tarif/kod değişmişse hata verir.
from yolox.exp import get_exp
for _m in ("nano_macro_long", "nano_macro"):   # get_exp import önbelleği: runtime içinde exp değiştiyse yeniden yükle
    sys.modules.pop(_m, None)
BATCH, FP16, CU_PER_HOUR = 64, True, 1.54
WORKERS = os.cpu_count()
_e = get_exp(EXP_FILE, None)
RECIPE_KEYS = ("max_epoch", "no_aug_epochs", "warmup_epochs", "warmup_lr", "basic_lr_per_img",
               "min_lr_ratio", "scheduler", "weight_decay", "momentum", "ema", "input_size",
               "random_size", "mosaic_prob", "mosaic_scale", "enable_mixup", "mixup_prob",
               "hsv_prob", "flip_prob", "degrees", "translate", "shear", "depth", "width",
               "num_classes", "train_ann", "val_ann", "nmsthre", "seed", "nas_snapshot_interval")
MAX_EPOCH, NO_AUG, SNAP = _e.max_epoch, _e.no_aug_epochs, _e.nas_snapshot_interval
PLAN = nl.eval_plan(MAX_EPOCH, SNAP)
manifest = ns.ensure_manifest(f"{DRIVE_RESULTS}/manifest.json", {
    "round": ROUND,
    "notebook": "notebooks/02d_nas_tur2_relu_tam_coco.ipynb",
    "decisions": "2. tur = ReLU k3 + kontrol, eşit uzun ince ayar, tam COCO ~12 epoch; karar ölçütü önceden "
                 "sabit (Furkan, 2026-10-12). Karar 5 (NAS en fazla 2 tur), Karar 2 (NAS bütçesi ≤ 1,0 puan).",
    "runs": list(nl.RUNS),
    "recipe": {**{k: getattr(_e, k, None) for k in RECIPE_KEYS}, "batch": BATCH, "fp16": FP16,
               "devices": 1, "init": "pretrained yolox_nano.pth", "in_training_eval": False,
               "train_images": N_TRAIN},
    "eval_protocol": {"split": "val2017 (5000)", "precision": "fp32", "batch": 64, "conf": 0.001,
                      "nms": _e.nmsthre,
                      "checkpoint": "EMA; ara epoch'lar epoch<N>_ckpt.pth, son epoch last_epoch_ckpt.pth",
                      "plan": [nl.eval_key(e, r) for e, r in PLAN]},
    "decision_rule": {"metric": "map5095", "resolution": nl.DECISION_RES, "epoch": MAX_EPOCH,
                      "delta": f"{nl.TREATMENT} − {nl.CONTROL}", "min_delta": nl.DECISION_MIN_DELTA,
                      "early_check_control_min_map": nl.EARLY_CHECK_MIN_MAP},
    "exp_file_sha256": ns.sha256(EXP_FILE),
    "base_exp_file_sha256": ns.sha256(BASE_EXP_FILE),
    "weights_sha256": env["weights_sha256"],
    "weights_url": env["weights_url"],
    "train_ann_sha256": env["train_ann_sha256"],
    "yolox_commit": env["yolox_commit"],
    "repo_commit_at_creation": env["repo_commit"],
    "cu_per_hour": CU_PER_HOUR,
    "env_at_creation": env,
}, keys=nl.FINGERPRINT_KEYS)
del _e
OUT_ROOT = f"{WS}/YOLOX_outputs/{ROUND}"

def stream(cmd, show, tag):
    return ns.stream(cmd, WS, show, tag)

print(json.dumps({k: manifest[k] for k in ("round", "created_utc", "runs", "recipe", "decision_rule")},
                 indent=1, default=str))
print("değerlendirme planı:", manifest["eval_protocol"]["plan"])

In [ ]:
# 7 · Doğrulama + koşu başına config.json
# (a) İki mimari kuruluyor ve önceden eğitilmiş ağırlık birebir yükleniyor mu (ReLU'da kaç SiLU değişti)?
# (b) Her koşunun çözülmüş exp ayarlarının tamamı + parametre/GFLOPs (çözünürlük başına) → config.json
import torch
from yolox.utils import get_model_info

def build(run_id, pretrained=WEIGHTS):
    for _m in ("nano_macro_long", "nano_macro"):
        sys.modules.pop(_m, None)
    e = get_exp(EXP_FILE, None)
    e.merge(ns.exp_opts(run_id, nas_pretrained=pretrained))
    return e, e.get_model().eval()

PLAIN = (int, float, str, bool, type(None), tuple, list)
for run_id in manifest["runs"]:
    e, m = build(run_id)
    assert e.nas_summary.get("pretrained_loaded"), e.nas_summary
    info = {}
    for r in ns.RESOLUTIONS:
        p, g = ns.INFO_RE.search(get_model_info(m, (r, r))).groups()
        info[r] = {"params_m": float(p), "gflops": float(g)}
    cfg = {"run_id": run_id, "arch_params": ns.arch_params(run_id), "surgery": e.nas_summary,
           "model_info": info, "exp": {k: v for k, v in vars(e).items() if isinstance(v, PLAIN)},
           "train_cmd_fixed": {"batch": BATCH, "fp16": FP16, "devices": 1}}
    ns.write_json(f"{DRIVE_RESULTS}/runs/{run_id}/config.json", cfg)
    print(f"{run_id:16s} {e.nas_summary} 416: {info[416]}")
del m

In [ ]:
# 8 · Duman testi (isteğe bağlı, ≈8–10 dk, ≈0,2 CU): uzun tarifin boru hattı, 10k alt kümede kısaltılmış
# ReLU kolu: 2 epoch (anlık görüntü her epoch) → kesinti benzetimi: 3. epoch --resume ile → epoch1
# anlık görüntüsünü 320'de değerlendir. Sonuçlar _smoke/ altına; turun kayıtlarına girmez.
import torch
RUN_SMOKE = True
if RUN_SMOKE:
    sid, smoke = nl.TREATMENT, f"{DRIVE_RESULTS}/_smoke"
    shutil.rmtree(f"{OUT_ROOT}/_smoke", ignore_errors=True); shutil.rmtree(smoke, ignore_errors=True)
    base = ["python", "-u", "YOLOX/tools/train.py", "-f", EXP_FILE, "-d", "1", "-b", str(BATCH), "--fp16",
            "-expn", "_smoke"]
    opts = ns.exp_opts(sid, nas_mirror_dir=smoke, output_dir=OUT_ROOT, data_num_workers=WORKERS,
                       no_aug_epochs=0, nas_snapshot_interval=1, train_ann="instances_train2017_10k.json")
    stream(base + opts + ["nas_pretrained", WEIGHTS, "max_epoch", "2"], ns.SHOW_TRAIN, "smoke 1-2")
    shutil.rmtree(f"{OUT_ROOT}/_smoke")                     # runtime sıfırlanmış gibi: yerel kopya yok
    os.makedirs(f"{OUT_ROOT}/_smoke")
    for n in ("latest_ckpt.pth", "train_log.txt"):
        shutil.copy(f"{smoke}/{n}", f"{OUT_ROOT}/_smoke/{n}")
    stream(base + ["--resume"] + opts + ["max_epoch", "3"], ns.SHOW_TRAIN, "smoke resume")
    log = open(f"{smoke}/train_log.txt").read()
    assert "resume training" in log and "start train epoch3" in log
    assert log.count("NAS adayı") == 2, "her denemede bir kez beklenir (logger iki kez kurulmuş olabilir)"
    for ep in (1, 2, 3):
        ck = torch.load(f"{smoke}/epoch{ep}_ckpt.pth", map_location="cpu")
        assert ck["start_epoch"] == ep, (ep, ck["start_epoch"])
    assert torch.load(f"{smoke}/last_epoch_ckpt.pth", map_location="cpu")["start_epoch"] == 3
    txt, s = stream(["python", "-u", "YOLOX/tools/eval.py", "-f", EXP_FILE, "-c", f"{smoke}/epoch1_ckpt.pth",
                     "-b", "64", "-d", "1", "--conf", "0.001", "--tsize", "320", "-expn", "_smoke_eval"]
                    + ns.exp_opts(sid, output_dir=OUT_ROOT), ns.SHOW_EVAL, "smoke eval")
    print("duman testi tamam: anlık görüntüler epoch1-3, devam, eval", ns.parse_eval(txt), f"eval {s:.0f} sn")

In [ ]:
# 9 · Eğitim döngüsü (kaldığı yerden devam eder) — iki kol, kontrol önce
# Durum: runs/<koşu>/status.json (pending → training → trained → evaluated). Checkpoint, log ve ara anlık
# görüntüler (epoch<N>_ckpt.pth) her epoch sonunda Drive'a atomik kopyalanır (exp'teki trainer kancası).
# Yarım koşu: Drive'daki latest_ckpt.pth yerele alınır, --resume ile devam edilir; checkpoint okunamazsa
# koşu baştan başlar. Devam ettirilen koşuda veri sırası/EMA ilk koşuyla birebir aynı değildir → attempts.
import torch
EVAL_ROOT = f"{WS}/YOLOX_outputs/{ROUND}_eval"

def ckpt_epoch(path):
    try:
        return torch.load(path, map_location="cpu")["start_epoch"]
    except Exception as e:
        print(f"  okunamayan checkpoint {path}: {e}")
        return None

def eval_one(run_id, epoch, res):
    """Tek (koşu, epoch, çözünürlük) değerlendirmesi; sonuç status.json'a yazılır, tekrar yapılmaz."""
    st = ns.get_status(DRIVE_RESULTS, run_id)
    key = nl.eval_key(epoch, res)
    if key in st["evals"]:
        return st
    mirror = f"{DRIVE_RESULTS}/runs/{run_id}"
    ckpt = f"{mirror}/{nl.ckpt_name(epoch, MAX_EPOCH)}"
    assert ckpt_epoch(ckpt) == epoch, f"{ckpt} epoch {epoch} değil"
    cmd = (["python", "-u", "YOLOX/tools/eval.py", "-f", EXP_FILE, "-c", ckpt,
            "-b", "64", "-d", "1", "--conf", "0.001", "--tsize", str(res),
            "-expn", f"{run_id}_{key}"] + ns.exp_opts(run_id, output_dir=EVAL_ROOT))
    txt, wall = stream(cmd, ns.SHOW_EVAL, f"{run_id} {key}")
    ev = ns.parse_eval(txt)
    assert ev["map5095"] is not None, "mAP okunamadı (eval.py hatası logger.catch ile yutulmuş olabilir)"
    tmp = f"{mirror}/eval_{key}_log.txt.tmp"
    open(tmp, "w").write(txt)
    os.replace(tmp, f"{mirror}/eval_{key}_log.txt")
    return ns.set_status(DRIVE_RESULTS, run_id, st,
                         evals={**st["evals"], key: {**ev, "epoch": epoch, "eval_s": round(wall, 1),
                                                     "evaluated_utc": ns.now_utc()}})

def train_run(run_id):
    st = ns.get_status(DRIVE_RESULTS, run_id)
    if st["status"] in ("trained", "evaluated"):
        return st
    local, mirror = f"{OUT_ROOT}/{run_id}", f"{DRIVE_RESULTS}/runs/{run_id}"
    shutil.rmtree(local, ignore_errors=True)
    os.makedirs(local)
    os.makedirs(mirror, exist_ok=True)
    ep = ckpt_epoch(f"{mirror}/latest_ckpt.pth") if os.path.exists(f"{mirror}/latest_ckpt.pth") else None
    if ep is None:   # baştan: yarım kalmış eski dosyalar (anlık görüntüler dahil) temizlenir
        for n in os.listdir(mirror):
            if n.endswith("_ckpt.pth") or n == "train_log.txt":
                os.remove(f"{mirror}/{n}")
    if ep is None or ep < MAX_EPOCH:
        resume = ep is not None
        if resume:
            for n in ("latest_ckpt.pth", "train_log.txt"):
                shutil.copy(f"{mirror}/{n}", f"{local}/{n}")
        attempt = {"start_utc": ns.now_utc(), "resume_from_epoch": ep, "gpu": env["gpu"],
                   "repo_commit": env["repo_commit"]}
        st = ns.set_status(DRIVE_RESULTS, run_id, st, status="training",
                           attempts=st["attempts"] + [attempt])
        cmd = (["python", "-u", "YOLOX/tools/train.py", "-f", EXP_FILE, "-d", "1", "-b", str(BATCH)]
               + (["--fp16"] if FP16 else []) + ["-expn", run_id] + (["--resume"] if resume else [])
               + ns.exp_opts(run_id, nas_pretrained="" if resume else WEIGHTS, nas_mirror_dir=mirror,
                             output_dir=OUT_ROOT, data_num_workers=WORKERS))
        _, wall = stream(cmd, ns.SHOW_TRAIN, run_id)
        st["attempts"][-1].update(end_utc=ns.now_utc(), wall_s=round(wall, 1))
    log = open(f"{mirror}/train_log.txt").read()
    ns.check_train_log(log, MAX_EPOCH, NO_AUG)
    assert ckpt_epoch(f"{mirror}/last_epoch_ckpt.pth") == MAX_EPOCH, "son epoch checkpoint'i eksik"
    for e, _ in PLAN:
        if e < MAX_EPOCH:
            assert ckpt_epoch(f"{mirror}/{nl.ckpt_name(e, MAX_EPOCH)}") == e, f"epoch{e} anlık görüntüsü eksik"
    train_s = round(sum(a.get("wall_s") or 0 for a in st["attempts"]), 1)
    return ns.set_status(DRIVE_RESULTS, run_id, st, status="trained", train_s=train_s,
                         trained_utc=ns.now_utc(), repo_commit=env["repo_commit"])

# Erken kontrol (1. turla aynı eşik): kontrol bitince son epoch 416'da değerlendirilir; eşiğin altındaysa tarif
# önceden eğitilmiş bilgiyi bozuyor demektir → ReLU kolu başlamaz (≈3 sa boşa harcanmaz).
for i, run_id in enumerate(manifest["runs"], 1):
    st = train_run(run_id)
    print(f"✔ {i}/{len(manifest['runs'])} {run_id}: {st['status']} · eğitim {st.get('train_s')} sn "
          f"· deneme {len(st['attempts'])}", flush=True)
    if run_id == nl.CONTROL:
        st = eval_one(run_id, MAX_EPOCH, nl.DECISION_RES)
        m = st["evals"][nl.eval_key(MAX_EPOCH, nl.DECISION_RES)]["map5095"]
        print(f"ERKEN KONTROL: kontrol {nl.DECISION_RES} mAP = {m} (orijinal Nano 25,8; 1. tur 24,5; "
              f"eşik {nl.EARLY_CHECK_MIN_MAP})", flush=True)
        assert m >= nl.EARLY_CHECK_MIN_MAP, "kontrol mAP eşiğin altında: tarif yeniden konuşulmalı, döngü durdu"

In [ ]:
# 10 · Değerlendirme döngüsü: her koşu × plan (ara epoch'lar 416, son epoch 320/416/512), fp32, val2017
# eval_one hücre 9'da tanımlı: kesintiden sonra önce hücre 9'u çalıştır (bitmiş koşuları hızla atlar).
def eval_run(run_id):
    st = ns.get_status(DRIVE_RESULTS, run_id)
    if st["status"] == "evaluated":
        return st
    assert st["status"] == "trained", f"{run_id} eğitilmemiş ({st['status']})"
    for e, r in PLAN:
        st = eval_one(run_id, e, r)
    return ns.set_status(DRIVE_RESULTS, run_id, st, status="evaluated")

for i, run_id in enumerate(manifest["runs"], 1):
    st = eval_run(run_id)
    print(f"✔ {i}/{len(manifest['runs'])} {run_id}: " +
          " · ".join(f"{k}: {v['map5095']}" for k, v in st["evals"].items()), flush=True)
    nl.rebuild_curve(DRIVE_RESULTS)

In [ ]:
# 11 · Özet: öğrenme eğrileri, fark (ReLU − kontrol), önceden sabitlenmiş karar, süre/CU
# Drive'a yazılır: curve.csv, nas_r2_summary.json, nas_r2_curve.png
import pandas as pd
rows = nl.rebuild_curve(DRIVE_RESULTS)
df = pd.DataFrame(rows)
print(df.pivot_table(index=["epoch", "resolution"], columns="run_id", values="map5095").to_string(), "\n")
print("Fark (ReLU − kontrol):\n", pd.DataFrame(nl.gaps(rows)).to_string(index=False), "\n")
dec = nl.decision(rows, MAX_EPOCH)
print("KARAR:", dec)

train_s = {r: ns.get_status(DRIVE_RESULTS, r).get("train_s") for r in manifest["runs"]}
eval_s = {k: round(float(v), 1) for k, v in df.groupby("run_id")["eval_s"].sum().items()}
cu = (sum(v or 0 for v in train_s.values()) + sum(eval_s.values())) / 3600 * CU_PER_HOUR
print(f"eğitim {train_s} sn · değerlendirme {eval_s} sn · CU ≈ {cu:.2f}")
ns.write_json(f"{DRIVE_RESULTS}/{ROUND}_summary.json", {
    "round": ROUND, "written_utc": ns.now_utc(), "decision": dec, "gaps": nl.gaps(rows),
    "train_s": train_s, "eval_s": eval_s, "cu_total": round(cu, 2),
    "attempts": {r: len(ns.get_status(DRIVE_RESULTS, r)["attempts"]) for r in manifest["runs"]},
})

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7, 4.5))
for run_id, g in df[df["resolution"] == nl.DECISION_RES].groupby("run_id"):
    g = g.sort_values("epoch")
    ax.plot(g["epoch"], g["map5095"], marker="o", label=run_id)
ax.axhline(25.8, ls=":", c="gray", lw=1, label="orijinal Nano (25,8)")
ax.set_xlabel("epoch (tam COCO train2017)"); ax.set_ylabel(f"mAP@[.5:.95] val2017, {nl.DECISION_RES}")
ax.set_title(f"2. tur: ReLU k3 vs kontrol · karar: {dec['status']}"); ax.legend()
fig.tight_layout(); fig.savefig(f"{DRIVE_RESULTS}/{ROUND}_curve.png", dpi=150)
plt.show()
print("Drive'a yazıldı:", sorted(f for f in os.listdir(DRIVE_RESULTS) if f.startswith(ROUND) or f.endswith(".csv")))